# Now ingest the documents 

## Import the Required Modules


## i am commenting the reference notebooks here but don't comment while we actually run this notebook

In [0]:
# %run /RAG_Project/Setup_env

In [0]:
# %run "/RAG_Project/Setup_Models& Vector_store"


In [0]:
from langchain_community.vectorstores import AzureSearch

vector_store = AzureSearch(
    azure_search_endpoint=vector_store_endpoint,
    azure_search_key=vector_store_admin_key,
    index_name=vector_store_name,
    embedding_function=embedding.embed_query
)

In [0]:
 #%pip install pypdf unstructured python-docx beautifulsoup4 lxml 
 #%pip install langchain_community 
 #%pip install docx2txt 
 #%pip install langchain_community

In [0]:

import os
from langchain_community.document_loaders import (
    PyPDFLoader,
    UnstructuredHTMLLoader,
    Docx2txtLoader,
    TextLoader,
)

In [0]:
volume_path = "/Volumes/rag_project/documents/azurechatbot"
files = dbutils.fs.ls(volume_path)
for f in files:
    print(f.path, f.size)

## Creating the loader function before reading the doc to handle any kind of file

In [0]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    UnstructuredHTMLLoader,
    Docx2txtLoader,
    TextLoader
)
import os
chunked_docs=[]


def load_document(file_path):
    # file_path is passed as an argument to this function
    ext = os.path.splitext(file_path)[1].lower()
    
    if ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext in [".html", ".htm"]:
        loader = UnstructuredHTMLLoader(file_path)
    elif ext == ".docx":
        loader = Docx2txtLoader(file_path)
    elif ext == ".txt":
        loader = TextLoader(file_path)
    else:
        raise ValueError(f"Unsupported file extension: {ext}")
    
    return loader.load()

## Reading the files from Databricks Volume

### Step-1 Extract Text from the documents

In [0]:

all_documents = [] 
#chunked_docs=[]
for f in files:
    file_path = f.path.replace("dbfs:", "")
    print(f"Loading: {file_path}")
    try:
        if file_path not in chunked_docs: #--> Making sure i am not loading the file again and again
            docs = load_document(file_path) #--> Here we are calling the load_Document Function
            all_documents.extend(docs)
            chunked_docs.append(file_path)
            print(f"  -> {len(docs)} documents loaded")
        else:
            print("files are loaded already, no new files exist")
    except Exception as e:
        print(f"  -> FAILED: {e}")

print(f"\nTotal documents loaded: {len(all_documents)}")

### Step-2 Convert text into chunks

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150,separators=["\n\n", "\n",".", " ", ""])
chunks = text_splitter.split_documents(all_documents)
print(len(chunks))


## ![Step](path)-3 Push the Chunks in Azure search AI Vector Store

### Generating Chunk_id for the each chunk 

In [0]:
import hashlib

def generate_chunk_id(chunk):
    source = chunk.metadata.get("source", "unknown")
    page = chunk.metadata.get("page", "na")
    content_hash = hashlib.sha256(chunk.page_content.encode("utf-8")).hexdigest()[:16]
    raw_id = f"{source}_p{page}_{content_hash}"
    return hashlib.sha256(raw_id.encode("utf-8")).hexdigest()

## Pushing chunks into Azure vector Store


In [0]:
#%pip install -U azure-search-documents

In [0]:
#dbutils.library.restartPython()

### Commenting the code there so space in vector store

/*

import time

batch_size = 100
total = len(chunks)  

for i in range(0, total, batch_size):
    batch = chunks[i:i + batch_size]  
    batch_ids = [generate_chunk_id(doc) for doc in batch]

    attempt = 0
    while attempt < 3:
        try:
            vector_store.add_documents(documents=batch, ids=batch_ids)
            print(f"Uploaded batch {i}-{i + len(batch)} of {total}")
            break
        except Exception as e:
            attempt += 1
            wait = 10 * attempt
            print(f"  Batch {i} failed (attempt {attempt}): {e}")
            print(f"  Retrying in {wait}s...")
            time.sleep(wait)
    else:
        print(f"  Batch {i} FAILED after 3 attempts - skipping")

print("Done.") 

## To Check the index of the vector_store

In [0]:
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

index_client = SearchIndexClient(
    endpoint=vector_store_endpoint,
    credential=AzureKeyCredential(vector_store_admin_key)
)

index = index_client.get_index(vector_store_name)
for field in index.fields:
    print(field.name, "-", field.type)

## Now we stored the embeddings in vector_store

in this process we changed the index of the vector store 

we add meta data as a new field in index schema

then we update the index using client index

then we push the embeddings into vector store

%md
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import SimpleField, SearchFieldDataType
from azure.core.credentials import AzureKeyCredential

index_client = SearchIndexClient(
    endpoint=vector_store_endpoint,
    credential=AzureKeyCredential(vector_store_admin_key)
)

### Fetch current index definition
index = index_client.get_index(vector_store_name)

###  Add the metadata field if it doesn't already exist
existing_field_names = [f.name for f in index.fields]
if "metadata" not in existing_field_names:
    index.fields.append(
        SimpleField(name="metadata", type=SearchFieldDataType.String)
    )
    index_client.create_or_update_index(index)
    print("Added 'metadata' field to index.")
else:
    print("'metadata' field already exists.")

###  Confirm
updated_index = index_client.get_index(vector_store_name)
for field in updated_index.fields:
    print(field.name, "-", field.type)